# Source Code:
*Note: Since mpi4py po yung gamit, kailangan po ng terminal command so yung output po for this is nasa second cell*

In [ ]:
import pandas as pd
import time
from mpi4py import MPI

# Staging Parallel
comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

# Start Time
start_time = time.time()

# ====================================================

# Lists of time per task
s_timepertask = [] # Sequential
p_timepertask = [] # Parallel

# Record when processing is done
s_timedone = 0
p_timedone = 0

# Sequential Processing
def sequential():
    seq_start = time.time()
    
    # Staging DataFrame (20 million rows > 5 million rows)
    boc2019_df = pd.read_csv(r'C:\Users\Josh\Downloads\boc_lite_2019.csv', encoding='latin-1', nrows=100000)
    read_time = time.time() - start_time
    print(f'DataFrame created in {read_time:.4f} seconds.\n\n')

    seq_start = seq_start - p_timedone # Decrease start time when parallel is already done
    seq_start = seq_start - read_time # Decrease start time from read time

    print('=== Sequential Processing ===\n\n\n')

        # Filter Data : Filter only 'CHN' as country import
    print('> Task 1: Filter Data\n')
    s_filtered_df = boc2019_df[boc2019_df['countryorigin_iso3'] == 'CHN']
    print(s_filtered_df.head(5))

    s_filterdata_time = time.time()
    s_filterdata_seconds = s_filterdata_time - seq_start
    s_timepertask.append(s_filterdata_seconds)
    print(f'\n> Task done in {s_filterdata_seconds:.4f} seconds.\n')
    print('==============')

        # Aggregate Data : Find the average dutiestaxes per country
    print('> Task 2: Aggregate Data\n')
    s_aggregated_df = boc2019_df.groupby('countryorigin_iso3').agg({'dutiablevaluephp': 'mean'}).reset_index()
    print(s_aggregated_df)

    s_aggdata_time = time.time()
    s_aggdata_seconds = s_aggdata_time - s_filterdata_time
    s_timepertask.append(s_aggdata_seconds)
    print(f'\n> Task done in {s_aggdata_seconds:.4f} seconds.\n')
    print('==============')

        # Sort Data : Sort dutiablevaluephp in descending order
    print('Task 3: Sort Data\n')
    s_sorted_df = boc2019_df.sort_values(by=['dutiablevaluephp'], ascending=False)
    print(s_sorted_df.head(5))

    s_sortdata_time = time.time()
    s_sortdata_seconds = s_sortdata_time - s_aggdata_time
    s_timepertask.append(s_sortdata_seconds)
    print(f'\n> Task done in {s_sortdata_seconds:.4f} seconds.\n')
    print('==============\n')

    s_timedone = sum(s_timepertask)
    print(f'Sequential Processing Time Total: {s_timedone:.4f} seconds.')

# Parallel Processing
def parallel():
    par_start = time.time()
    par_start = par_start - s_timedone # Decrease start time when parallel is already done

    # Distribution Logic =========
    if rank == 0:  # Master
        print('=== Parallel Processing ===\n\n\n')

        # Staging DataFrame (20 million rows > 5 million rows)
        boc2019_df = pd.read_csv(r'C:\Users\Josh\Downloads\boc_lite_2019.csv', encoding='latin-1', nrows=100000)
        read_time = time.time() - start_time
        print(f'DataFrame created in {read_time:.4f} seconds.\n\n')

        # Split dataset into chunks
        total_rows = len(boc2019_df)
        rows_per_process = total_rows // (size - 1)  # Exclude master from work
        
        # Send chunks to workers
        for i in range(1, size):
            start_idx = (i - 1) * rows_per_process
            end_idx = start_idx + rows_per_process if i < size - 1 else total_rows
            chunk = boc2019_df.iloc[start_idx:end_idx]
            comm.send(chunk, dest=i)
        
        # Task 1: Filter Data
        task1_start = time.time()
        filtered_results = []
        for i in range(1, size):
            result = comm.recv(source=i)
            filtered_results.append(result)
        
        p_filtered_df = pd.concat(filtered_results, ignore_index=True)
        p_filter_time = time.time() - task1_start
        p_timepertask.append(p_filter_time)
        print('> Task 1: Filter Data\n')
        print(p_filtered_df.head(5))
        print(f'\n> Task done in {p_filter_time:.4f} seconds.\n')
        print('==============')
        
        # Task 2: Aggregate Data
        task2_start = time.time()
        agg_results = []
        for i in range(1, size):
            result = comm.recv(source=i)
            agg_results.append(result)
        
        p_aggregated_df = pd.concat(agg_results).groupby(level=0).mean().reset_index()
        p_agg_time = time.time() - task2_start
        p_timepertask.append(p_agg_time)
        print('> Task 2: Aggregate Data\n')
        print(p_aggregated_df)
        print(f'\n> Task done in {p_agg_time:.4f} seconds.\n')
        print('==============')
        
        # Task 3: Sort Data
        task3_start = time.time()
        sorted_results = []
        for i in range(1, size):
            result = comm.recv(source=i)
            sorted_results.append(result)
        
        p_sorted_df = pd.concat(sorted_results, ignore_index=True).sort_values(by=['dutiablevaluephp'], ascending=False)
        p_sort_time = time.time() - task3_start
        p_timepertask.append(p_sort_time)
        print('Task 3: Sort Data\n')
        print(p_sorted_df.head(5))
        print(f'\n> Task done in {p_sort_time:.4f} seconds.\n')
        print('==============\n')
        
        p_timedone = sum(p_timepertask)
        print(f'Parallel Processing Time Total: {p_timedone:.4f} seconds.')
        
    else:  # Workers
        # Receive chunk from master
        local_df = comm.recv(source=0)
        
        # Task 1: Filter
        local_filtered = local_df[local_df['countryorigin_iso3'] == 'CHN']
        comm.send(local_filtered, dest=0)
        print(f'Processor {rank} has done Task 1')
        
        # Task 2: Aggregate
        local_agg = local_df.groupby('countryorigin_iso3').agg({'dutiablevaluephp': 'mean'})
        comm.send(local_agg, dest=0)
        print(f'Processor {rank} has done Task 2')
        
        # Task 3: Sort
        local_sorted = local_df.sort_values(by=['dutiablevaluephp'], ascending=False)
        comm.send(local_sorted, dest=0)
        print(f'Processor {rank} has done Task 3')


# ====================================================

def main():
    if rank == 0:
        sequential()
    
    comm.Barrier()  # Wait for sequential to finish before parallel starts
    
    parallel()

    if rank == 0:
        end_time = time.time()
        print(f'{end_time - start_time:.4f} seconds in total.')
main()

## **Output:**

In [7]:
import subprocess
result = subprocess.run(
    ["mpiexec", "-n", "5", 
     r"C:\Users\Josh\Desktop\Josh Folder\School\Perpetual\3rd Year\Parallel and Distributed Computing\.venv\Scripts\python.exe",
     r"C:\Users\Josh\Desktop\Josh Folder\School\Perpetual\3rd Year\Parallel and Distributed Computing\PDC-Mid-Proj1-MendozaJoshLeonard.py"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)



Processor 1 has done Task 1
Processor 1 has done Task 2
Processor 1 has done Task 3
Processor 2 has done Task 1
Processor 2 has done Task 2
Processor 2 has done Task 3
Processor 3 has done Task 1
Processor 3 has done Task 2
Processor 3 has done Task 3
Processor 4 has done Task 1
Processor 4 has done Task 2
Processor 4 has done Task 3
DataFrame created in 0.6663 seconds.


=== Sequential Processing ===



> Task 1: Filter Data

               uid    ty      tq  ... countryexport_iso3  subport  port
0  201901 00000001  2019  2019q1  ...                CHN      NaN   NaN
1  201901 00000002  2019  2019q1  ...                CHN      NaN   NaN
4  201901 00000005  2019  2019q1  ...                CHN      NaN   NaN
5  201901 00000006  2019  2019q1  ...                CHN      NaN   NaN
7  201901 00000008  2019  2019q1  ...                CHN      NaN   NaN

[5 rows x 30 columns]

> Task done in 1.4150 seconds.

> Task 2: Aggregate Data

    countryorigin_iso3  dutiablevaluephp
0             

## **Table:**

In [15]:
import re
import pandas as pd

seq_times = re.findall(r'Task done in ([\d.]+) seconds', result.stdout[:result.stdout.find('=== Parallel')])
par_times = re.findall(r'Task done in ([\d.]+) seconds', result.stdout[result.stdout.find('=== Parallel'):])

tasks = ['Filter Data', 'Aggregate Data', 'Sort Data']

summary_df = pd.DataFrame({
    'Task': tasks,
    'Sequential (seconds)': [float(t) for t in seq_times],
    'Parallel (seconds)': [float(t) for t in par_times],
})

totals = pd.DataFrame([{
    'Task': 'TOTAL',
    'Sequential (seconds)': summary_df['Sequential (seconds)'].sum(),
    'Parallel (seconds)': summary_df['Parallel (seconds)'].sum(),
}])

summary_df = pd.concat([summary_df, totals], ignore_index=True)
summary_df


,Task,Sequential (seconds),Parallel (seconds)
0,Filter Data,1.4150,0.0939
1,Aggregate Data,0.0275,0.0030
2,Sort Data,0.0588,0.3165
3,TOTAL,1.5013,0.4134


### **Technical Report**

1. **Problem description**
- The primary objective of the program was to optimize the processing of a dataset (Bureau of Customs 2019 lite data) using distributed computing and measure whether a parallelized approach using multiple processors could significantly reduce execution time compared to a standard sequential Python script. The activity focused on three specific data operations:
 - Filtering: Extracting records where the country of origin is "CHN".
 - Aggregation: Calculating the mean dutiablevaluephp grouped by country.
 - Sorting: Organizing the dataset by dutiable value in descending order.
 
2. **Parallelization approach**
- The parallelization approach of the program utilizes the Master-Worker architecture using mpi4py with 5 processes. The master loads the primary DataFrame, partitions the data into chunks, distributes tasks to workers, and gathers the results. The ffour other workers receive data chunks from the master, perform local filtering, aggregation, and sorting, then send the processed results back to the master.

3. **Performance analysis**

| Task | Sequential (seconds) | Parallel (seconds) |
| ---- | -------------------- | ------------------ |
| 0	| Filter Data | 1.4150 | 0.0939 |
| 1	| Aggregate Data	| 0.0275	| 0.0030 |
| 2	| Sort Data	| 0.0588	| 0.3165 |
| 3 	| TOTAL	| 1.5013	| 0.4134 |

- Based on the summary of the execution logs, the parallel implementation showed a substantial overall performance gain with Parallel being faster than Sequential on the Filter and Aggregate functions, while showing that Parallel is slower than Sequential in the sorting function. This is likely due to communication overhead, where the master has to send chunks to 4 workers, then receive 4 sorted chunks back via MPI. The serialization and deserialization of DataFrames over MPI takes more time than just doing it sequentially. There is also a factor where when the master is receiving all sorted chunks, is still has to pd.concat + sort_values again on the combined result to get a globally sorted dataset.


4. **Challenges encountered**
- There are a few challenges that I encountered. First, running mpi4py requires a specific terminal environment and the mpiexec command, which cannot be executed directly within standard Jupyter cells without using subprocess or external scripts. Second, it was a bit clunky making the transition between the sequential and parallel function, so the comm.Barrier() was necessary to make sure that the sequential baseline was finished before the parallel processes began.